# FLUX Autonomous Phase Embedding

**Purpose:** Embed the new `phase_autonomous` and updated `phase_orchestrator` code into Flux-Apex-V1.flx

**New Components Being Added:**
1. **phase_autonomous/** — Complete autonomous agent system
   - `tool_executor.py` — Central tool dispatcher (28 built-in tools)
   - `code_sandbox.py` — Safe Python execution (20 allowed modules)
   - `coder_pool.py` — Parallel sandbox execution via coder model
   - `document_ingester.py` — Multi-format document processing
   - `goal_planner.py` — Proactive goal and planning system
   - `autonomous_agent.py` — Main agent tying all components together

2. **phase_orchestrator/** updates
   - `native_json_orchestrator.py` — Qwen2.5 native JSON function calling

3. **phase_claw/** updates  
   - `flux_bridge.py` — Updated with autonomous integration

**Version Update:** 8.2-fixed-imports → 8.3-autonomous

**Target Environment:** Google Colab

In [ ]:
# Cell 1: Environment Setup (Colab-optimized)
import os
import sys
import gc
import gzip
import base64
import shutil
from pathlib import Path
from datetime import datetime

# Detect environment
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ON_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")

# Clone/pull FLUX repo
if ON_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git /content/FLUX
    else:
        !git -C /content/FLUX pull
elif ON_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git /kaggle/working/FLUX
    else:
        !git -C /kaggle/working/FLUX pull
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_model.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_model.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(f"Root: {ROOT}")
print(f"PyTorch available: ", end="")
try:
    import torch
    print(f"✓ {torch.__version__}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
except ImportError:
    print("✗ Installing...")
    !pip install -q torch

print("✓ Environment configured")

In [ ]:
# Cell 2: HuggingFace Token & Model Download
import torch
from huggingface_hub import hf_hub_download

# Get HF token
HF_TOKEN = None
if ON_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN from Colab secrets")
    except:
        pass
elif ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN from Kaggle secrets")
    except:
        pass

if not HF_TOKEN:
    try:
        from flux_utils import _resolve_hf_token
        HF_TOKEN = _resolve_hf_token()
        if HF_TOKEN:
            print("✓ HF_TOKEN from environment")
    except:
        pass

if not HF_TOKEN:
    print("⚠ No HF_TOKEN - may need for upload. Set in Colab secrets.")
else:
    os.environ['HF_TOKEN'] = HF_TOKEN

# Download model
MODEL_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'
MODEL_PATH.parent.mkdir(exist_ok=True)

if not MODEL_PATH.exists():
    print("\n📥 Downloading Flux-Apex-V1.flx from HuggingFace...")
    print("   (This may take 5-10 minutes)")
    hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=HF_TOKEN,
    )
    print(f"✓ Downloaded: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print(f"✓ Model exists: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")

In [ ]:
# Cell 3: Define NEW files to embed

# NEW phase_autonomous files (complete autonomous system)
NEW_AUTONOMOUS_FILES = [
    'phases/phase_autonomous/__init__.py',
    'phases/phase_autonomous/tool_executor.py',
    'phases/phase_autonomous/code_sandbox.py',
    'phases/phase_autonomous/coder_pool.py',
    'phases/phase_autonomous/document_ingester.py',
    'phases/phase_autonomous/goal_planner.py',
    'phases/phase_autonomous/autonomous_agent.py',
]

# UPDATED phase_orchestrator files (native JSON calling)
UPDATED_ORCHESTRATOR_FILES = [
    'phases/phase_orchestrator/native_json_orchestrator.py',
]

# UPDATED phase_claw files (autonomous integration)
UPDATED_CLAW_FILES = [
    'phases/phase_claw/flux_bridge.py',
    'phases/phase_claw/__init__.py',
]

ALL_NEW_FILES = NEW_AUTONOMOUS_FILES + UPDATED_ORCHESTRATOR_FILES + UPDATED_CLAW_FILES

print(f"Files to add/update: {len(ALL_NEW_FILES)}")
print(f"  New autonomous: {len(NEW_AUTONOMOUS_FILES)}")
print(f"  Updated orchestrator: {len(UPDATED_ORCHESTRATOR_FILES)}")
print(f"  Updated claw: {len(UPDATED_CLAW_FILES)}")

# Verify files exist
missing = []
for f in ALL_NEW_FILES:
    path = ROOT / f
    if not path.exists():
        missing.append(f)
        print(f"  ✗ Missing: {f}")
    else:
        print(f"  ✓ {f} ({path.stat().st_size} bytes)")

if missing:
    print(f"\n⚠ {len(missing)} files missing - make sure repo is up to date!")
    print("  Run: git pull in the FLUX directory")

In [ ]:
# Cell 4: Collect and validate files

def compress_source(source: str) -> str:
    """Compress source code using gzip + base64."""
    compressed = gzip.compress(source.encode('utf-8'))
    return base64.b64encode(compressed).decode('ascii')

def decompress_source(compressed: str) -> str:
    """Decompress source code."""
    return gzip.decompress(base64.b64decode(compressed)).decode('utf-8')

def collect_files(file_list: list, root: Path) -> tuple:
    """Collect file contents, validate syntax."""
    collected = {}
    errors = []
    
    for file_path in file_list:
        full_path = root / file_path
        
        if not full_path.exists():
            print(f"⚠ Missing: {file_path}")
            continue
            
        source = full_path.read_text(encoding='utf-8')
        
        # Validate syntax
        try:
            compile(source, file_path, 'exec')
            collected[file_path] = source
            lines = source.count('\n') + 1
            print(f"✓ {file_path} ({lines} lines)")
        except SyntaxError as e:
            errors.append((file_path, str(e)))
            print(f"✗ {file_path} - SYNTAX ERROR: {e}")
    
    return collected, errors

print("Collecting new files...\n")
new_code, syntax_errors = collect_files(ALL_NEW_FILES, ROOT)

if syntax_errors:
    print(f"\n✗ {len(syntax_errors)} syntax errors - FIX BEFORE CONTINUING")
    for f, e in syntax_errors:
        print(f"  {f}: {e}")
else:
    print(f"\n✓ All {len(new_code)} files validated")

# Compute stats
new_lines = sum(src.count('\n') + 1 for src in new_code.values())
new_raw_bytes = sum(len(src.encode('utf-8')) for src in new_code.values())
print(f"\nNew code statistics:")
print(f"  Files: {len(new_code)}")
print(f"  Lines: {new_lines:,}")
print(f"  Raw size: {new_raw_bytes:,} bytes ({new_raw_bytes/1024:.1f} KB)")

In [ ]:
# Cell 5: Load current model

print(f"Loading {MODEL_PATH}...")
print("  (This may take 1-2 minutes for 16GB file)")

flx = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)

print(f"\n✓ Loaded model")
print(f"  Version: {flx.get('version', 'unknown')}")
print(f"  Format: {flx.get('format', 'unknown')}")
print(f"  Phase: {flx.get('phase', 'unknown')}")

# Check existing runtime
if 'runtime' in flx:
    runtime = flx['runtime']
    existing_files = len(runtime.get('code', {}))
    existing_version = runtime.get('version', 'unknown')
    print(f"\n  Existing runtime:")
    print(f"    Version: {existing_version}")
    print(f"    Files: {existing_files}")
    
    # Check if phase_autonomous already embedded
    existing_code = runtime.get('code', {})
    autonomous_present = any('phase_autonomous' in k for k in existing_code.keys())
    print(f"    phase_autonomous present: {'Yes ⚠' if autonomous_present else 'No'}")
else:
    print(f"\n  No existing runtime - will create new")

In [ ]:
# Cell 6: Merge new files into runtime

# Get existing code bundle
if 'runtime' in flx and 'code' in flx['runtime']:
    existing_code = flx['runtime']['code']
    print(f"Existing runtime has {len(existing_code)} files")
else:
    existing_code = {}
    print("No existing runtime - creating fresh")

# Compress new files
print("\nCompressing new files...")
new_compressed = {}
for path, source in new_code.items():
    compressed = compress_source(source)
    new_compressed[path] = compressed
    ratio = len(source) / len(compressed) if compressed else 0
    print(f"  {path}: {len(source)} → {len(compressed)} bytes ({ratio:.1f}x)")

# Merge (new files override existing)
merged_code = {**existing_code}  # Start with existing
update_count = 0
add_count = 0

for path, compressed in new_compressed.items():
    if path in merged_code:
        update_count += 1
        print(f"  📝 Updating: {path}")
    else:
        add_count += 1
        print(f"  ➕ Adding: {path}")
    merged_code[path] = compressed

print(f"\n✓ Merged: {add_count} added, {update_count} updated")
print(f"  Total files in runtime: {len(merged_code)}")

In [ ]:
# Cell 7: Update runtime and metadata

# Calculate new statistics
total_lines = 0
total_raw = 0
for path, compressed in merged_code.items():
    source = decompress_source(compressed)
    total_lines += source.count('\n') + 1
    total_raw += len(source.encode('utf-8'))

compressed_size = sum(len(c) for c in merged_code.values())

# Create updated runtime
new_runtime = {
    'version': '8.3-autonomous',
    'embedded_at': datetime.now().isoformat(),
    'bootstrap': flx.get('runtime', {}).get('bootstrap', ''),  # Preserve existing bootstrap
    'code': merged_code,
    'metadata': {
        'total_files': len(merged_code),
        'total_lines': total_lines,
        'raw_size_bytes': total_raw,
        'compressed_size_bytes': compressed_size,
        'compression_ratio': total_raw / compressed_size if compressed_size > 0 else 0,
        'tiers': {
            'tier1_critical': len([f for f in merged_code if 'phase1' in f or 'phase2' in f or 'phase3' in f or 'phase4' in f or 'phase5' in f or 'phase6' in f or 'phase8' in f]),
            'tier2_orchestration': len([f for f in merged_code if 'orchestrator' in f or 'unified' in f or 'phase9_arc' in f]),
            'tier3_multimodal': len([f for f in merged_code if 'phase8_9' in f]),
            'tier4_claw': len([f for f in merged_code if 'claw' in f]),
            'tier5_autonomous': len([f for f in merged_code if 'autonomous' in f]),
        },
        'new_in_8.3': [
            'phase_autonomous/tool_executor.py — 28 built-in tools',
            'phase_autonomous/code_sandbox.py — Safe Python with 20 modules',
            'phase_autonomous/coder_pool.py — Parallel sandbox execution (Jules-style)',
            'phase_autonomous/document_ingester.py — Multi-format ingestion',
            'phase_autonomous/goal_planner.py — Proactive planning system',
            'phase_autonomous/autonomous_agent.py — Main agent coordinator',
            'phase_orchestrator/native_json_orchestrator.py — Qwen2.5 JSON calling',
        ],
    },
}

# Update bootstrap if missing
if not new_runtime['bootstrap']:
    bootstrap_path = ROOT / 'bootstrap.py'
    if bootstrap_path.exists():
        new_runtime['bootstrap'] = bootstrap_path.read_text()
        print("✓ Added bootstrap.py")

# Update model
flx['runtime'] = new_runtime
flx['version'] = '8.3-autonomous'
flx['phase'] = 'complete'
flx['timestamp'] = datetime.now().isoformat()

# Update metadata
if 'metadata' not in flx:
    flx['metadata'] = {}
flx['metadata']['last_modified'] = datetime.now().isoformat()
flx['metadata']['modified_components'] = list(set(
    flx['metadata'].get('modified_components', []) + ['runtime']
))

# Add capabilities
caps = flx['metadata'].get('capabilities', [])
new_caps = ['autonomous_agent', 'coder_pool', 'native_json_tools', 'document_ingestion', 'goal_planning']
for cap in new_caps:
    if cap not in caps:
        caps.append(cap)
flx['metadata']['capabilities'] = caps

print(f"\n✓ Runtime updated")
print(f"  Version: {new_runtime['version']}")
print(f"  Files: {len(merged_code)}")
print(f"  Lines: {total_lines:,}")
print(f"  Size: {compressed_size/1024:.1f} KB compressed")

In [ ]:
# Cell 8: Save updated model (with disk space handling)

# Check disk space
_, _, disk_free = shutil.disk_usage(MODEL_PATH.parent)
disk_free_gb = disk_free / 1e9
model_size_gb = MODEL_PATH.stat().st_size / 1e9 if MODEL_PATH.exists() else 0

print(f"Disk space check:")
print(f"  Free: {disk_free_gb:.1f} GB")
print(f"  Model: {model_size_gb:.2f} GB")
print(f"  Needed: ~{model_size_gb + 2:.1f} GB minimum")

# CRITICAL: On Colab with limited disk, delete original first
# The model is safely in memory (flx variable)
if disk_free_gb < model_size_gb + 5:
    print(f"\n⚠ Low disk space - deleting original before save...")
    print(f"   (Model is safe in memory)")
    if MODEL_PATH.exists():
        os.remove(MODEL_PATH)
        gc.collect()
        print(f"   ✓ Freed {model_size_gb:.2f} GB")

# Save
print(f"\n💾 Saving to {MODEL_PATH}...")
print(f"   This may take 2-5 minutes...")

torch.save(flx, str(MODEL_PATH), pickle_protocol=4)

new_size = MODEL_PATH.stat().st_size / 1e9
print(f"\n✓ Saved successfully!")
print(f"  Size: {new_size:.2f} GB")
print(f"  Version: {flx['version']}")

In [ ]:
# Cell 9: Verify bootstrap works

print("Verifying bootstrap with new files...\n")

# Clear cached modules to force reload
for key in list(sys.modules.keys()):
    if key.startswith('phases.'):
        del sys.modules[key]

# Test bootstrap
try:
    from bootstrap import wake_up
    
    result = wake_up(str(MODEL_PATH), device='cpu', verbose=True)
    
    print(f"\n{'='*60}")
    print("BOOTSTRAP VERIFICATION")
    print(f"{'='*60}")
    
    modules_loaded = len(result.get('modules', {}))
    print(f"Modules loaded: {modules_loaded}")
    
    # Check specific new modules
    modules = result.get('modules', {})
    new_modules = [
        'phases.phase_autonomous',
        'phases.phase_autonomous.tool_executor',
        'phases.phase_autonomous.code_sandbox',
        'phases.phase_autonomous.coder_pool',
        'phases.phase_autonomous.autonomous_agent',
        'phases.phase_orchestrator.native_json_orchestrator',
    ]
    
    print(f"\nNew modules check:")
    for mod in new_modules:
        status = '✓' if mod in modules else '✗'
        print(f"  {status} {mod}")
    
    all_present = all(mod in modules for mod in new_modules)
    print(f"\n{'✓ All new modules loaded!' if all_present else '⚠ Some modules missing'}")
    
except Exception as e:
    print(f"✗ Bootstrap failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 10: Quick functional test

print("Quick functional test of new components...\n")

try:
    # Test imports
    from phases.phase_autonomous import (
        FluxToolExecutor,
        CodeSandbox,
        CoderPool,
        DocumentIngester,
        GoalPlanner,
        AutonomousAgent,
    )
    print("✓ All autonomous imports work")
    
    from phases.phase_orchestrator.native_json_orchestrator import (
        NativeJSONOrchestrator,
        load_native_tools,
    )
    print("✓ Native JSON orchestrator imports work")
    
    # Quick test of CodeSandbox
    sandbox = CodeSandbox(timeout=5)
    result = sandbox.execute("print('Hello from embedded sandbox!')")
    print(f"✓ Sandbox test: {result.output.strip()}")
    
    # Quick test of CoderPool
    pool = CoderPool({}, max_sandboxes=1)
    print(f"✓ CoderPool created with {pool.max_sandboxes} sandbox(es)")
    pool.shutdown()
    
    print("\n✓ All functional tests passed!")
    
except Exception as e:
    print(f"✗ Test failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Cell 11: Upload to HuggingFace

UPLOAD_TO_HF = True  # Set to False to skip upload

if UPLOAD_TO_HF and HF_TOKEN:
    from huggingface_hub import HfApi
    
    print("📤 Uploading to HuggingFace Hub...")
    print("   This may take 10-30 minutes for 16GB file...")
    
    api = HfApi(token=HF_TOKEN)
    
    try:
        api.upload_file(
            path_or_fileobj=str(MODEL_PATH),
            path_in_repo='checkpoints/Flux-Apex-V1.flx',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'v8.3-autonomous: Add phase_autonomous (CoderPool, GoalPlanner, DocumentIngester, native JSON tools)',
        )
        print("\n✓ Uploaded successfully!")
        print("  Repo: https://huggingface.co/UnseenGAP/FLUX")
        
    except Exception as e:
        print(f"\n✗ Upload failed: {e}")
        print("\nTo upload manually later:")
        print("  from huggingface_hub import HfApi")
        print("  api = HfApi(token=YOUR_TOKEN)")
        print("  api.upload_file(...)")
        
elif UPLOAD_TO_HF and not HF_TOKEN:
    print("⚠ Upload requested but no HF_TOKEN available")
    print("  Set HF_TOKEN in Colab secrets to enable upload")
    
else:
    print("ℹ Upload skipped (UPLOAD_TO_HF = False)")
    print("  Set UPLOAD_TO_HF = True in Cell 11 to upload")

In [ ]:
# Cell 12: Final Summary

print(f"""
{'='*70}
           FLUX AUTONOMOUS PHASE EMBEDDING COMPLETE
{'='*70}

📦 Model: {MODEL_PATH.name}
📏 Size:  {MODEL_PATH.stat().st_size / 1e9:.2f} GB
🏷️  Version: {flx['version']}

📂 Runtime Statistics:
   Files: {len(merged_code)}
   Lines: {total_lines:,}
   Size:  {compressed_size/1024:.1f} KB compressed

🆕 New in v8.3-autonomous:
┌────────────────────────────────────────────────────────────────────┐
│ phase_autonomous/                                                  │
│   ├── tool_executor.py      28 built-in tools dispatched          │
│   ├── code_sandbox.py       Safe Python (20 allowed modules)      │
│   ├── coder_pool.py         Parallel sandboxes (Jules-style)      │
│   ├── document_ingester.py  PDF, DOCX, MD, JSON, CSV, HTML        │
│   ├── goal_planner.py       Proactive multi-step planning         │
│   └── autonomous_agent.py   Main agent coordinator                │
│                                                                    │
│ phase_orchestrator/                                                │
│   └── native_json_orchestrator.py  Qwen2.5 JSON function calling  │
└────────────────────────────────────────────────────────────────────┘

🏗️ Architecture:
   ┌─────────────────────────────────────────────────────────────┐
   │         INSTRUCT MODEL (Brain) - Qwen2.5-1.5B-Instruct      │
   │                          │                                   │
   │              delegate_to_coder()                            │
   │                          ↓                                   │
   │  ┌──────────────────────────────────────────────────────┐  │
   │  │  CODER POOL — Qwen2.5-Coder-1.5B-Instruct           │  │
   │  │  ┌────────┐ ┌────────┐ ┌────────┐ ┌────────┐        │  │
   │  │  │Sandbox1│ │Sandbox2│ │Sandbox3│ │Sandbox4│        │  │
   │  │  └────────┘ └────────┘ └────────┘ └────────┘        │  │
   │  └──────────────────────────────────────────────────────┘  │
   └─────────────────────────────────────────────────────────────┘

✅ Instruct model stays free for reasoning
✅ Coder handles all code generation + execution
✅ Multiple sandboxes run in parallel

{'='*70}
""")

# Cleanup
del flx
gc.collect()
print("✓ Memory cleaned up")